In [ ]:
# Qwen 3.8 27B Abliterated — Kaggle, uma célula
# Usa llama.cpp pré-compilado. O mmproj extra ativa a entrada de imagens no Qwen.
import os, re, sys, time, signal, secrets, pathlib, shutil, subprocess, threading, urllib.request

try:
    import requests
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests"], check=True)
    import requests
try:
    from huggingface_hub import hf_hub_download
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub[hf_xet]"], check=True)
    from huggingface_hub import hf_hub_download

MODEL = "huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF:UD-Q4_K_XL"
REPO = "huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF"
MMPROJ_FILE = "mmproj-model-bf16.gguf"
MMPROJ_DIR = "/kaggle/working/qwen-mmproj"
PORT = 8000
LOCAL_URL = f"http://127.0.0.1:{PORT}"
LLAMA = os.path.join(os.path.expanduser("~"), ".local", "bin", "llama")
API_KEY_FILE = "/kaggle/working/qwen-api-key.txt"
SERVER_LOG = "/kaggle/working/llama-server.log"
TUNNEL_LOG = "/kaggle/working/cloudflared.log"

subprocess.run(["nvidia-smi"], check=True)
if not os.path.isfile(LLAMA):
    print("A instalar llama.cpp pré-compilado com deteção de CUDA...")
    result = subprocess.run("curl -LsSf https://llama.app/install.sh | sh", shell=True,
        executable="/bin/bash", stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    if result.returncode != 0 or not os.path.isfile(LLAMA):
        raise RuntimeError("Instalação do llama.cpp falhou. Confirma a Internet no Kaggle.")
else:
    print("llama.cpp pré-compilado já está instalado.")
os.environ["PATH"] = os.path.dirname(LLAMA) + os.pathsep + os.environ.get("PATH", "")

if os.path.isfile(API_KEY_FILE):
    API_KEY = pathlib.Path(API_KEY_FILE).read_text().strip()
else:
    API_KEY = secrets.token_urlsafe(32)
    pathlib.Path(API_KEY_FILE).write_text(API_KEY)
    os.chmod(API_KEY_FILE, 0o600)

# O projetor tem cerca de 931 MB; huggingface_hub mostra a barra de progresso.
print("A verificar/descarregar o projetor multimodal mmproj para imagens…")
MMPROJ_PATH = hf_hub_download(repo_id=REPO, filename=MMPROJ_FILE, local_dir=MMPROJ_DIR)


def api_pronta():
    try:
        if not requests.get(LOCAL_URL + "/health", timeout=3).ok:
            return False
        r = requests.get(LOCAL_URL + "/v1/models",
            headers={"Authorization": "Bearer " + API_KEY}, timeout=5)
        return r.ok and any("qwen" in str(m.get("id", "")).lower() for m in r.json().get("data", []))
    except Exception:
        return False


def servidor_com_mmproj():
    for pasta in pathlib.Path("/proc").iterdir():
        if not pasta.name.isdigit():
            continue
        try:
            cmdline = (pasta / "cmdline").read_bytes()
            if LLAMA.encode() in cmdline and b"serve" in cmdline and str(PORT).encode() in cmdline:
                return b"--mmproj" in cmdline and MMPROJ_PATH.encode() in cmdline
        except (FileNotFoundError, PermissionError, ProcessLookupError):
            pass
    return False


def terminar_servidor_antigo():
    for pasta in pathlib.Path("/proc").iterdir():
        if not pasta.name.isdigit():
            continue
        try:
            cmdline = (pasta / "cmdline").read_bytes()
            if LLAMA.encode() in cmdline and b"serve" in cmdline and str(PORT).encode() in cmdline:
                os.kill(int(pasta.name), signal.SIGTERM)
        except (FileNotFoundError, PermissionError, ProcessLookupError):
            pass
    time.sleep(3)


def mostrar_saida(pipe, log_path):
    try:
        with open(log_path, "a", encoding="utf-8", errors="replace") as log:
            buffer = ""
            while True:
                char = pipe.read(1)
                if not char:
                    break
                if char in ("\n", "\r"):
                    if buffer.strip():
                        print(buffer, flush=True)
                        log.write(buffer + "\n"); log.flush()
                    buffer = ""
                else:
                    buffer += char
            if buffer.strip():
                print(buffer, flush=True); log.write(buffer + "\n"); log.flush()
    except Exception as erro:
        print("Leitor do log:", erro, flush=True)


def iniciar_servidor():
    print("\nA iniciar o Qwen com visão. A saída abaixo mostra o carregamento.\n")
    pathlib.Path(SERVER_LOG).write_text("")
    processo = subprocess.Popen([
        LLAMA, "serve", "-hf", MODEL,
        "--mmproj", MMPROJ_PATH,
        "--host", "0.0.0.0", "--port", str(PORT),
        "--alias", "qwen3.8-27b-abliterated", "--api-key", API_KEY,
        "--ctx-size", "4096", "--gpu-layers", "all", "--tensor-split", "1,1",
        "--jinja", "--parallel", "1", "--no-webui"
    ], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
    threading.Thread(target=mostrar_saida, args=(processo.stdout, SERVER_LOG), daemon=True).start()
    for segundo in range(900):
        if api_pronta():
            print("\n✅ Qwen pronto com suporte multimodal.")
            return processo
        if processo.poll() is not None:
            raise RuntimeError("O servidor terminou; consulta " + SERVER_LOG)
        if segundo and segundo % 30 == 0:
            print(f"A carregar o modelo há {segundo//60} min {segundo%60} s…", flush=True)
        time.sleep(1)
    raise TimeoutError("Qwen não ficou pronto em 15 minutos; consulta a saída e " + SERVER_LOG)


# Não reutiliza uma instância antiga de texto que não tenha carregado o mmproj.
if api_pronta() and servidor_com_mmproj():
    print("O servidor Qwen com mmproj já está pronto; vou reutilizá-lo.")
    server_process = None
else:
    terminar_servidor_antigo()
    server_process = iniciar_servidor()

if not shutil.which("cloudflared"):
    print("A instalar cloudflared…")
    deb = "/kaggle/working/cloudflared.deb"
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb", deb)
    subprocess.run(["dpkg", "-i", deb], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


def iniciar_tunel():
    pathlib.Path(TUNNEL_LOG).write_text("")
    log = open(TUNNEL_LOG, "a", buffering=1)
    processo = subprocess.Popen(["cloudflared", "tunnel", "--url", LOCAL_URL, "--no-autoupdate"],
        stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(120):
        texto = pathlib.Path(TUNNEL_LOG).read_text(errors="ignore")
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", texto)
        if match:
            return processo, match.group(0)
        if processo.poll() is not None:
            break
        time.sleep(2)
    raise RuntimeError("O túnel não abriu:\n" + pathlib.Path(TUNNEL_LOG).read_text(errors="ignore")[-3000:])


def model_id():
    try:
        r = requests.get(LOCAL_URL + "/v1/models", headers={"Authorization": "Bearer " + API_KEY}, timeout=15)
        r.raise_for_status()
        return r.json()["data"][0]["id"]
    except Exception:
        return "qwen3.8-27b-abliterated"


def mostrar_config():
    print("\n✅ Qwen ligado ao Kaggle")
    print("URL base:", public_url + "/v1")
    print("ID do modelo:", MODEL_ID)
    print("Chave API:", API_KEY)
    print("Mega Harness → Definições → Fornecedores")
    print("Mantém o URL e a chave privados.")


tunnel_process, public_url = iniciar_tunel()
MODEL_ID = model_id()
mostrar_config()

# Mantém a célula ativa e reinicia servidor/túnel se algum parar.
while True:
    try:
        if not api_pronta():
            print("\nServidor Qwen deixou de responder; a reiniciar…")
            terminar_servidor_antigo()
            server_process = iniciar_servidor()
            MODEL_ID = model_id()
        if tunnel_process.poll() is not None:
            print("\nTúnel parado; a criar outro URL…")
            tunnel_process, public_url = iniciar_tunel()
            mostrar_config()
    except Exception as erro:
        print("Monitorização:", erro, flush=True)
    time.sleep(30)
